https://colab.research.google.com/github/cs221m/cs221m-course/blob/main/03_behavioral_analysis.ipynb

In [1]:
from IPython.display import clear_output

In [2]:
from datasets import load_dataset, concatenate_datasets # hf datasets

subjects = ['abstract_algebra', 'high_school_mathematics', 'college_mathematics']

N_SHOTS = 5

ds_list = []
dev_examples = {}

for subj in subjects:
    subj_ds = load_dataset('cais/mmlu', subj, split='test')
    
    subj_ds = subj_ds.map(lambda _: {'subject': subj}, batched=False) # create cleaner dict
    ds_list.append(subj_ds)
    
    subj_dev = load_dataset('cais/mmlu', subj, split='validation')
    dev_examples[subj] = list(subj_dev.select(range(min(N_SHOTS, len(subj_dev))))) # select few examples as dict
    
    print(f"    {subj}: {len(subj_ds)} test questions, {len(dev_examples[subj])} dev shots")
    
ds = concatenate_datasets(ds_list)

print(f"\nTotal: {len(ds)} test questions across {len(subjects)} subjects")
print(f'Few-shot: {N_SHOTS} dev examples per subject')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


    abstract_algebra: 100 test questions, 5 dev shots
    high_school_mathematics: 270 test questions, 5 dev shots
    college_mathematics: 100 test questions, 5 dev shots

Total: 470 test questions across 3 subjects
Few-shot: 5 dev examples per subject


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_id = 'microsoft/Phi-3.5-mini-instruct'
revision = 'main'

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    padding_side='left', # not important
)
tokenizer.pad_token = tokenizer.eos_token # use <eos> as padding

model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=torch.half,
            device_map='auto',
            revision=revision # download main branch
        ).eval() # inference mode 

clear_output()

In [5]:
LABELS = ['A', 'B', 'C', 'D']

def format_mmlu_prompt(question, choices, subject, labels=LABELS, few_shot_examples=None):
    subject_str = subject.replace("_", " ")
    prompt = f"The following are multiple choice questioons (with answers) about {subject_str}.\n\n"
    
    if few_shot_examples:
        for ex in few_shot_examples:
            prompt += ex['question'] + '\n'
            for i, choice in enumerate(ex['choices']):
                prompt += f'{labels[i]}. {choice}\n'
            prompt += f'Answer: {labels[ex['answer']]}\n\n'
            
    prompt += question + '\n'
    for i, choice in enumerate(choices):
        prompt += f'{labels[i]}. {choice}\n'
    prompt += "Answer:"
    return prompt

def predict(model, tokenizer, prompts):
    label_token_ids = [tokenizer.encode(label, add_special_tokens=False)[0] for label in LABELS]
    
    inputs = tokenizer(prompts, return_tensors='pt', padding='longest', padding_side='left').to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    
    last_logits = logits[torch.arange(len(prompts)), -1]
    
    answer_logits = last_logits[:, label_token_ids]
    predictions = answer_logits.argmax(dim=-1).cpu().tolist()
    return predictions

In [6]:
example_prompt = format_mmlu_prompt(
    ds[0]['question'], ds[0]['choices'], ds[0]['subject'],
    few_shot_examples=dev_examples[ds[0]['subject']]
)
print(example_prompt)

The following are multiple choice questioons (with answers) about abstract algebra.

The cyclic subgroup of Z_24 generated by 18 has order
A. 4
B. 8
C. 12
D. 6
Answer: A

Find the order of the factor group Z_6/<3>.
A. 2
B. 3
C. 6
D. 12
Answer: B

Statement 1 | A permutation that is a product of m even permutations and n odd permutations is an even permutation if and only if n is even. Statement 2 | Every group is isomorphic to a group of permutations.
A. True, True
B. False, False
C. True, False
D. False, True
Answer: A

Find the order of the factor group (Z_4 x Z_12)/(<2> x <2>)
A. 2
B. 3
C. 4
D. 12
Answer: C

Find the maximum possible order for some element of Z_4 x Z_6.
A. 4
B. 6
C. 12
D. 24
Answer: C

Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
A. 0
B. 4
C. 2
D. 6
Answer:


In [8]:
from tqdm import tqdm
from collections import defaultdict

BATCH_SIZE = 1

prompts = [
    format_mmlu_prompt(ex['question'], ex['choices'], ex['subject'],
                        few_shot_examples=dev_examples[ex['subject']])
    for ex in ds
]
gold_labels = [ex['answer'] for ex in ds]
example_subjects = [ex['subject'] for ex in ds]

all_preds = []
for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc='Evaluating'):
    batch = prompts[i: i + BATCH_SIZE]
    preds = predict(model, tokenizer, batch)
    all_preds.extend(preds)

Evaluating: 100%|██████████| 470/470 [02:23<00:00,  3.29it/s]
